# Transformeri

## 1. Provera GPU-a

In [2]:
import torch

torch.cuda.is_available()

True

## 2. Instalacija

In [3]:
!pip -q install "transformers>=4.40" datasets accelerate sentence-transformers

import random

import numpy as np
import torch
from transformers import set_seed

SEME = 42

random.seed(SEME)
np.random.seed(SEME)
torch.manual_seed(SEME)
set_seed(SEME)

## 3. Podaci


In [4]:
from google.colab import files

_ = files.upload()

Saving news_colab.parquet to news_colab.parquet


In [5]:
import pandas as pd

df = pd.read_parquet("news_colab.parquet")

print(len(df), "clanaka")
print(df["label"].value_counts())

38514 clanaka
label
0    21189
1    17325
Name: count, dtype: int64


## 4. Podela na trening i test


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text_ok"], df["label"], test_size=0.25,
    stratify=df["label"], random_state=42)

rezultati = {}

len(X_train), len(X_test)

(28885, 9629)

## 5. Dotreniranje DistilBERT-a


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

MODEL = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)


def tokenizuj(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)


ds_train = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
ds_test = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

ds_train = ds_train.map(tokenizuj, batched=True, remove_columns=["text"])
ds_test = ds_test.map(tokenizuj, batched=True, remove_columns=["text"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/28885 [00:00<?, ? examples/s]

Map:   0%|          | 0/9629 [00:00<?, ? examples/s]

In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score


def metrike(eval_pred):
    logiti, labele = eval_pred
    pred = np.argmax(logiti, axis=1)

    e = np.exp(logiti - logiti.max(axis=1, keepdims=True))
    verovatnoce = (e / e.sum(axis=1, keepdims=True))[:, 1]

    preciznost, odziv, f1, _ = precision_recall_fscore_support(
        labele, pred, average="binary", zero_division=0)

    return {"accuracy": accuracy_score(labele, pred),
            "precision": preciznost,
            "recall": odziv,
            "f1": f1,
            "roc_auc": roc_auc_score(labele, verovatnoce)}

In [9]:
import inspect
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    kljuc = {"processing_class": tokenizer}
else:
    kljuc = {"tokenizer": tokenizer}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=metrike,
    **kljuc,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.064896,0.014342,0.995846,0.998837,0.991919,0.995366,0.999763
2,0.006347,0.011161,0.997300,0.998149,0.995844,0.996995,0.999750


TrainOutput(global_step=1806, training_loss=0.025865036123746785, metrics={'train_runtime': 360.7518, 'train_samples_per_second': 160.138, 'train_steps_per_second': 5.006, 'total_flos': 3826320810178560.0, 'train_loss': 0.025865036123746785, 'epoch': 2.0})

In [10]:
ocena = trainer.evaluate()

rezultati["DistilBERT"] = {
    "accuracy": ocena["eval_accuracy"],
    "precision": ocena["eval_precision"],
    "recall": ocena["eval_recall"],
    "f1": ocena["eval_f1"],
    "roc_auc": ocena["eval_roc_auc"],
}

rezultati["DistilBERT"]

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.006347,0.011161,2,0.997300,0.998149,0.995844,0.996995,0.999750


{'accuracy': 0.9972998234499948,
 'precision': 0.9981485767183522,
 'recall': 0.9958439159547449,
 'f1': 0.9969949144706426,
 'roc_auc': 0.9997502575435036}

## 6. Embedinzi kao ulaz u klasifikator


In [11]:
from sentence_transformers import SentenceTransformer

koder = SentenceTransformer("all-MiniLM-L6-v2",
                            device="cuda" if torch.cuda.is_available() else "cpu")

E_train = koder.encode(X_train.tolist(), batch_size=128, show_progress_bar=True)
E_test = koder.encode(X_test.tolist(), batch_size=128, show_progress_bar=True)

E_train.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/226 [00:00<?, ?it/s]

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

(28885, 384)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

modeli = {"Embedinzi + LogReg": LogisticRegression(max_iter=2000),
          "Embedinzi + LinearSVC": LinearSVC()}

for ime, m in modeli.items():
    m.fit(E_train, y_train)
    pred = m.predict(E_test)

    preciznost, odziv, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="binary", zero_division=0)

    rezultati[ime] = {"accuracy": accuracy_score(y_test, pred),
                      "precision": preciznost,
                      "recall": odziv,
                      "f1": f1}

    if hasattr(m, "predict_proba"):
        rezultati[ime]["roc_auc"] = roc_auc_score(y_test, m.predict_proba(E_test)[:, 1])

    print(ime)
    print(classification_report(y_test, pred, target_names=["true", "fake"], digits=4))

Embedinzi + LogReg
              precision    recall  f1-score   support

        true     0.9383    0.9536    0.9459      5298
        fake     0.9420    0.9233    0.9326      4331

    accuracy                         0.9400      9629
   macro avg     0.9402    0.9385    0.9392      9629
weighted avg     0.9400    0.9400    0.9399      9629

Embedinzi + LinearSVC
              precision    recall  f1-score   support

        true     0.9473    0.9558    0.9515      5298
        fake     0.9454    0.9349    0.9401      4331

    accuracy                         0.9464      9629
   macro avg     0.9463    0.9454    0.9458      9629
weighted avg     0.9464    0.9464    0.9464      9629



## 7. Rezultati


In [13]:
import json

with open("rezultati_colab.json", "w") as f:
    json.dump(rezultati, f, indent=2, default=float)

files.download("rezultati_colab.json")

pd.DataFrame(rezultati).T.round(4)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,accuracy,precision,recall,f1,roc_auc
DistilBERT,0.9973,0.9981,0.9958,0.9970,0.9998
Embedinzi + LogReg,0.9400,0.9420,0.9233,0.9326,0.9839
Embedinzi + LinearSVC,0.9464,0.9454,0.9349,0.9401,NaN
